# 00 — Bayesian Inference for State Estimation

> **Module:** `00-foundations/probabilities-estimation`  
> **Level:** Foundational  
> **Time:** ~2 hours  

---

## Why this matters

Every filter in this roadmap — Kalman, EKF, Particle — is a special case of one idea:

$$P(\text{state} \mid \text{measurement}) \propto P(\text{measurement} \mid \text{state}) \cdot P(\text{state})$$

This is Bayes' theorem. If you understand this equation *intuitively*, the rest of the roadmap will feel like engineering — not magic.

---

## Learning objectives

By the end of this notebook you will be able to:

1. State and apply Bayes' theorem for continuous distributions
2. Explain the role of prior, likelihood, and posterior in estimation
3. Implement a 1D Bayesian estimator from scratch
4. Understand why Gaussian distributions are computationally convenient
5. Connect Bayesian inference to the Kalman filter update step


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

---
## Part 1 — Bayes' theorem: the core idea

### Setting

Imagine you are a drone at an unknown altitude. You have:
- A **prior belief** about your altitude (from your last estimate)
- A **measurement** from your barometer (noisy)

Bayes' theorem tells you how to combine them:

$$\underbrace{P(h \mid z)}_{\text{posterior}} \propto \underbrace{P(z \mid h)}_{\text{likelihood}} \cdot \underbrace{P(h)}_{\text{prior}}$$

where $h$ is altitude and $z$ is the barometer reading.

### Key vocabulary

| Term | Symbol | Meaning |
|------|--------|---------|
| **Prior** | $P(h)$ | What you believed *before* the measurement |
| **Likelihood** | $P(z \mid h)$ | How probable the measurement is, *given* the state |
| **Posterior** | $P(h \mid z)$ | Updated belief *after* incorporating the measurement |
| **Evidence** | $P(z)$ | Normalisation constant (often ignored) |


In [ ]:
# --- Gaussian Bayesian update (1D) ---
# We will fuse a prior belief with a measurement, both modelled as Gaussians.
# This is exactly what the Kalman filter update step does.

h = np.linspace(80, 120, 500)  # altitude range in metres

# Prior: we think we are at 100m, with std = 5m
prior_mean, prior_std = 100.0, 5.0
prior = norm.pdf(h, prior_mean, prior_std)

# Measurement: barometer reads 108m, with std = 3m (sensor noise)
meas_mean, meas_std = 108.0, 3.0
likelihood = norm.pdf(h, meas_mean, meas_std)

# Posterior: proportional to prior × likelihood
posterior_unnorm = prior * likelihood
posterior = posterior_unnorm / (np.trapz(posterior_unnorm, h))  # normalise

# Analytical solution for Gaussian×Gaussian
posterior_var = 1 / (1/prior_std**2 + 1/meas_std**2)
posterior_mean = posterior_var * (prior_mean/prior_std**2 + meas_mean/meas_std**2)
posterior_std = np.sqrt(posterior_var)

print(f"Prior:     mean={prior_mean:.1f}m, std={prior_std:.1f}m")
print(f"Measurement: mean={meas_mean:.1f}m, std={meas_std:.1f}m")
print(f"Posterior: mean={posterior_mean:.2f}m, std={posterior_std:.2f}m")
print(f"\n→ The posterior is BETWEEN prior and measurement (weighted by precision)")

In [ ]:
fig, ax = plt.subplots()

ax.plot(h, prior, 'b--', linewidth=2, label=f'Prior  μ={prior_mean}, σ={prior_std}')
ax.plot(h, likelihood, 'g--', linewidth=2, label=f'Likelihood  μ={meas_mean}, σ={meas_std}')
ax.plot(h, posterior, 'r-', linewidth=2.5, label=f'Posterior  μ={posterior_mean:.1f}, σ={posterior_std:.2f}')

ax.axvline(prior_mean, color='blue', alpha=0.3, linestyle=':')
ax.axvline(meas_mean, color='green', alpha=0.3, linestyle=':')
ax.axvline(posterior_mean, color='red', alpha=0.5, linestyle=':')

ax.set_xlabel('Altitude (m)')
ax.set_ylabel('Probability density')
ax.set_title('Bayesian update: fusing prior belief with measurement')
ax.legend()
plt.tight_layout()
plt.show()

print("\nObservations:")
print("1. The posterior is narrower than both prior and likelihood → fusion REDUCES uncertainty")
print("2. The posterior mean is pulled toward the MORE PRECISE distribution")
print("3. This is EXACTLY what the Kalman filter update step computes")

---
## Part 2 — Why Gaussians are special

The product of two Gaussians is another Gaussian. This is the key property that makes the Kalman filter analytically tractable.

**Gaussian product formula:**

If $p_1 = \mathcal{N}(\mu_1, \sigma_1^2)$ and $p_2 = \mathcal{N}(\mu_2, \sigma_2^2)$, then:

$$\mu_{\text{posterior}} = \frac{\mu_1 / \sigma_1^2 + \mu_2 / \sigma_2^2}{1/\sigma_1^2 + 1/\sigma_2^2}$$

$$\sigma^2_{\text{posterior}} = \frac{1}{1/\sigma_1^2 + 1/\sigma_2^2}$$

Equivalently, defining **precision** $\lambda = 1/\sigma^2$:

$$\lambda_{\text{posterior}} = \lambda_1 + \lambda_2 \quad \text{(precisions add)}$$

> **Insight:** Adding a measurement always increases precision (reduces uncertainty). This is the fundamental guarantee of Bayesian fusion.


In [ ]:
def gaussian_product(mu1, var1, mu2, var2):
    """Fuse two Gaussian beliefs. Returns (mean, variance) of posterior."""
    var_post = 1 / (1/var1 + 1/var2)
    mu_post = var_post * (mu1/var1 + mu2/var2)
    return mu_post, var_post


# Simulate sequential fusion: 5 barometer measurements of the same altitude
true_altitude = 105.0  # metres (unknown to the filter)
sensor_std = 4.0

# Start with a vague prior
mu, var = 100.0, 25.0

history = [(mu, np.sqrt(var))]

np.random.seed(42)
for i in range(8):
    z = true_altitude + np.random.randn() * sensor_std  # noisy measurement
    mu, var = gaussian_product(mu, var, z, sensor_std**2)
    history.append((mu, np.sqrt(var)))

means, stds = zip(*history)
steps = list(range(len(means)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(steps, means, 'ro-', linewidth=2, label='Posterior mean')
ax1.fill_between(steps,
                 [m - s for m, s in zip(means, stds)],
                 [m + s for m, s in zip(means, stds)],
                 alpha=0.2, color='red', label='±1σ')
ax1.axhline(true_altitude, color='black', linestyle='--', label=f'True altitude={true_altitude}m')
ax1.set_xlabel('Measurement index')
ax1.set_ylabel('Altitude (m)')
ax1.set_title('Sequential Bayesian estimation')
ax1.legend()

ax2.plot(steps, stds, 'bs-', linewidth=2)
ax2.set_xlabel('Measurement index')
ax2.set_ylabel('Posterior std (m)')
ax2.set_title('Uncertainty decreases with each measurement')
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

---
## Part 3 — Multivariate Gaussians

In real navigation, the state has multiple dimensions: position in X and Y, velocity, heading...

The scalar formulas generalise directly:

$$\mathbf{x} \sim \mathcal{N}(\boldsymbol{\mu}, \mathbf{P})$$

where:
- $\boldsymbol{\mu} \in \mathbb{R}^n$ — mean vector
- $\mathbf{P} \in \mathbb{R}^{n \times n}$ — covariance matrix (symmetric, positive definite)

The covariance matrix encodes:
- **Diagonal terms** $P_{ii}$: variance of each state component
- **Off-diagonal terms** $P_{ij}$: correlation between components


In [ ]:
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

def plot_covariance_ellipse(ax, mean, cov, n_std=1.0, **kwargs):
    """Draw a covariance ellipse on axes ax."""
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(eigenvalues)
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ellipse)


fig, axes = plt.subplots(1, 3, figsize=(15, 5))

cases = [
    {"cov": np.array([[4.0, 0.0], [0.0, 4.0]]),   "title": "Uncorrelated, equal variance"},
    {"cov": np.array([[9.0, 0.0], [0.0, 1.0]]),   "title": "Uncorrelated, different variance"},
    {"cov": np.array([[4.0, 3.0], [3.0, 4.0]]),   "title": "Correlated (off-diagonal ≠ 0)"},
]

for ax, case in zip(axes, cases):
    mean = np.array([0.0, 0.0])
    cov = case["cov"]
    samples = np.random.multivariate_normal(mean, cov, 300)
    
    ax.scatter(samples[:, 0], samples[:, 1], alpha=0.3, s=10, color='steelblue')
    plot_covariance_ellipse(ax, mean, cov, n_std=1, fill=False, edgecolor='red', linewidth=2, label='1σ')
    plot_covariance_ellipse(ax, mean, cov, n_std=2, fill=False, edgecolor='orange', linewidth=1.5, label='2σ')
    
    ax.set_xlim(-7, 7)
    ax.set_ylim(-7, 7)
    ax.set_aspect('equal')
    ax.set_title(case["title"], fontsize=11)
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')
    ax.legend(fontsize=9)
    ax.text(0.05, 0.95, f'P = {cov}', transform=ax.transAxes,
            fontsize=8, verticalalignment='top', fontfamily='monospace')

plt.suptitle('Covariance matrix geometry — the shape of uncertainty', fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 4 — Connection to the Kalman Filter

We now have all the conceptual pieces. The Kalman filter is a **recursive Bayesian estimator** for linear Gaussian systems.

At each time step it does two things:

1. **Predict** — propagate the distribution forward in time using the motion model:
$$P(\mathbf{x}_k) = \int P(\mathbf{x}_k \mid \mathbf{x}_{k-1}) \cdot P(\mathbf{x}_{k-1}) \, d\mathbf{x}_{k-1}$$

2. **Update** — apply Bayes' theorem with the new measurement:
$$P(\mathbf{x}_k \mid \mathbf{z}_k) \propto P(\mathbf{z}_k \mid \mathbf{x}_k) \cdot P(\mathbf{x}_k)$$

For linear Gaussian systems, both operations have **closed-form solutions** — the Gaussian stays Gaussian throughout. This is what makes the Kalman filter exact and efficient.

**The equations** (preview — full derivation in Module 01):

| Step | Equation | Interpretation |
|------|----------|----------------|
| Predict mean | $\hat{\mathbf{x}}^- = \mathbf{A}\hat{\mathbf{x}}$ | Propagate state with dynamics |
| Predict cov | $\mathbf{P}^- = \mathbf{A}\mathbf{P}\mathbf{A}^T + \mathbf{Q}$ | Propagate uncertainty + add process noise |
| Kalman gain | $\mathbf{K} = \mathbf{P}^-\mathbf{H}^T(\mathbf{H}\mathbf{P}^-\mathbf{H}^T + \mathbf{R})^{-1}$ | Weight: model vs sensor precision |
| Update mean | $\hat{\mathbf{x}} = \hat{\mathbf{x}}^- + \mathbf{K}(\mathbf{z} - \mathbf{H}\hat{\mathbf{x}}^-)$ | Bayesian posterior mean |
| Update cov | $\mathbf{P} = (\mathbf{I} - \mathbf{K}\mathbf{H})\mathbf{P}^-$ | Bayesian posterior covariance |

> Module `01-kalman-filter` derives every one of these equations from first principles.


---
## Exercises

### Exercise 1 — Manual Bayesian update

You are estimating the distance to a building.
- Your prior: $\mu_0 = 50$ m, $\sigma_0 = 8$ m
- Laser rangefinder measurement: $z = 62$ m, $\sigma_z = 3$ m

**Q1:** Compute the posterior mean and standard deviation analytically.  
**Q2:** Plot prior, likelihood, and posterior on the same figure.  
**Q3:** Which distribution dominates the posterior, and why?


In [ ]:
# Exercise 1 — your solution here
prior_mean_ex, prior_std_ex = 50.0, 8.0
meas_mean_ex, meas_std_ex = 62.0, 3.0

# TODO: compute posterior
post_mean_ex, post_var_ex = None, None  # replace with your computation

# TODO: plot


### Exercise 2 — Sequential fusion

You receive 10 GPS measurements of a stationary drone at true position $x = 37.4$ m.  
GPS noise: $\sigma = 5$ m. Prior: $\mu_0 = 30$ m, $\sigma_0 = 10$ m.

**Q1:** Simulate the sequential fusion using `gaussian_product`.  
**Q2:** Plot the evolution of posterior mean and standard deviation.  
**Q3:** After how many measurements does the std fall below 1 m?


In [ ]:
# Exercise 2 — your solution here
np.random.seed(0)
true_pos = 37.4
gps_std = 5.0
mu_ex2, var_ex2 = 30.0, 100.0

# TODO: simulate 10 measurements and fuse sequentially


---
## Summary

| Concept | Key takeaway |
|---------|-------------|
| Bayes' theorem | posterior ∝ likelihood × prior |
| Gaussian product | posterior is Gaussian; precision adds |
| Covariance matrix | shape of uncertainty in multiple dimensions |
| Sequential fusion | uncertainty shrinks monotonically with each measurement |
| Kalman filter | recursive Bayesian estimation for linear Gaussian systems |

**Next notebook:** `01-kalman-filter/theory/01_kf_derivation.ipynb` — full derivation of the Kalman filter.
